In [25]:
import os
import sys
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

from utils.summary import get_model_stats

import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [ ]:
import torch
from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR, MultiStepLR, SequentialLR, LinearLR, StepLR

class ConstantWithWarmup(torch.optim.lr_scheduler._LRScheduler):
    def __init__(
        self,
        optimizer,
        num_warmup_steps: int,
    ):
        self.num_warmup_steps = num_warmup_steps
        super().__init__(optimizer)

    def get_lr(self):
        if self._step_count <= self.num_warmup_steps:
            # warmup
            scale = 1.0 - (self.num_warmup_steps - self._step_count) / self.num_warmup_steps
            lr = [base_lr * scale for base_lr in self.base_lrs]
            self.last_lr = lr
        else:
            lr = self.base_lrs
        return lr

    
def lr_lambda(epoch, warmup_epochs=50):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs  # Linear warmup
    return 1.0

model = None
optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9)
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

# Define the warmup scheduler
warmup_steps = 500
warmup_scheduler = LambdaLR(optimizer, lr_lambda=lambda step: min(1.0, step / warmup_steps))

# Define the decay scheduler (e.g., MultiStepLR for step decay)
decay_epochs = [30, 60, 90]  # Epoch milestones for decay
gamma = 0.1  # Decay factor
decay_scheduler = MultiStepLR(optimizer, milestones=decay_epochs, gamma=gamma)

# Combine the schedulers using SequentialLR
schedulers = [warmup_scheduler, decay_scheduler]
milestones = [warmup_steps]  # Switch to decay after warmup_steps
scheduler = SequentialLR(optimizer, schedulers, milestones=milestones)

# Sequential 2
warmup = LinearLR(optimizer, start_factor=1e-3, total_iters=5)
main_sched = StepLR(optimizer, step_size=30, gamma=0.1)
scheduler = SequentialLR(optimizer, schedulers=[warmup, main_sched], milestones=[5])